In [4]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster

In [16]:
import pandas as pd
import numpy as np

In [ ]:
local=True
if local:
    cluster=LocalCluster(memory_limit='48G')
    client=Client(cluster)
else:
    cluster=SLURMCluster(
        cores=2,#cores per slurm job
        memory="32G",#memory per slurm job
        processes=1,#dask workers per slurm jobTrueT
        job_extra_directives=["-p day", 
            f"--job-name=simclust_worker",
            f"--time=3:00:00",
            f"--output=worker_%j.out"]
    )
    cluster.scale(jobs=4)
    client = Client(cluster,
            timeout=f"{5*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s"  # Worker heartbeat interval
        )

2025-12-04 15:51:50,879 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 48G due to system memory limit of 32.00 GiB
2025-12-04 15:51:50,880 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 48G due to system memory limit of 32.00 GiB
2025-12-04 15:51:50,881 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 48G due to system memory limit of 32.00 GiB
2025-12-04 15:51:50,883 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 48G due to system memory limit of 32.00 GiB


2025-12-04 17:24:45,989 - distributed.scheduler - WARNING - Worker failed to heartbeat for 589s; attempting restart: <WorkerState 'tcp://127.0.0.1:36089', name: 0, status: running, memory: 155, processing: 0>
2025-12-04 17:24:45,989 - distributed.scheduler - WARNING - Worker failed to heartbeat for 589s; attempting restart: <WorkerState 'tcp://127.0.0.1:41961', name: 1, status: running, memory: 103, processing: 0>
2025-12-04 17:24:45,989 - distributed.scheduler - WARNING - Worker failed to heartbeat for 589s; attempting restart: <WorkerState 'tcp://127.0.0.1:46061', name: 3, status: running, memory: 172, processing: 0>
2025-12-04 17:24:45,990 - distributed.scheduler - WARNING - Worker failed to heartbeat for 589s; attempting restart: <WorkerState 'tcp://127.0.0.1:46457', name: 2, status: running, memory: 125, processing: 0>
2025-12-04 17:24:46,010 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:46457' caused the cluster to lose already computed task(s), which will 

In [24]:
DATA_ROOT="/home/mcn26/project_pi_skr2/shared/tabula_data"
simpath="simulated/shendure_pow_analysis/sim_with_orthos_20251203"
o0=scm.ortho.load(client,f"{DATA_ROOT}/{simpath}/orthos","0")

In [14]:
mats=o0.by_cell_type_design["Cardiomyocytes"].result()

In [ ]:
def mom(matricies):
    """
    Helper function for `_tensorzinb_fit` implementing warm start method of moments
    or parameter initalization.
    See [[Fixing zinb initialization]] for math
    """
        # nb portion #
    undone_nb=scm.undo_one_hot_encoding(matricies["nb_regressors"])
    undone_nb=undone_nb.rename({"cre_id, contr.treatment(base=\'reference\')":'cre_id'},axis=1)

    working_nb=pd.DataFrame({
                'cre_id':undone_nb["cre_id"],
                'umis_mpra_bc':matricies["regressand"]["umis_mpra_bc"]
            })

    ## compute mean and n ##
    working_nb = (
        working_nb
        .groupby("cre_id")
        .agg(
            mean_umis_mpra_bc=('umis_mpra_bc', 'mean'),
            var_umis_mpra_bc=('umis_mpra_bc', 'var'),
            n=('umis_mpra_bc', 'count')
        )
    )
    
    ## compute nb betas ##
    
    ref=working_nb.loc["reference"]["mean_umis_mpra_bc"]
    working_nb["fc"]=working_nb["mean_umis_mpra_bc"]/ref
    working_nb["lfc"]=np.log(working_nb["fc"])
    working_nb["beta"]=working_nb["lfc"]
    working_nb["beta"].loc["reference"]=np.log(working_nb["mean_umis_mpra_bc"].loc["reference"])
    
    nb_betas=working_nb["beta"]#.to_numpy()
    #obs. note [[LFC is beta]] has proof
    
    # estimating ZI #
    ## gross zi : get the total number of zeroes in each replicate ##
    undone_zi=scm.undo_one_hot_encoding(matricies["zi_regressors"])
    
    working_zi=pd.DataFrame({
        'rep_id':undone_zi["rep_id"],
        'umis_mpra_bc':matricies["regressand"]["umis_mpra_bc"]
    })
    #indicator
    working_zi["ind"]=working_zi["umis_mpra_bc"]==0
    working_zi=working_zi.drop(columns=["umis_mpra_bc"])
    #"gross" because we are not estimating zeros contributed from NB
    zeros=working_zi.groupby("rep_id").agg(
        gross_zeros=("ind","mean")
    )
    
    ## estimate the number of zeroes from the nb portion ##
    working_nb["valid_nb"]=working_nb["var_umis_mpra_bc"] > working_nb["mean_umis_mpra_bc"]
    working_nb["p"]=working_nb["mean_umis_mpra_bc"]/working_nb["var_umis_mpra_bc"]
    working_nb["r"]=working_nb["mean_umis_mpra_bc"]**2 / (working_nb["var_umis_mpra_bc"] - working_nb["mean_umis_mpra_bc"])
    #initalize to nan
    working_nb["zero_prop"]=np.nan
    #fill out valid nb cases with zero proportion...
    working_nb.loc[working_nb["valid_nb"],"zero_prop"]=working_nb["p"]**working_nb["r"]
    #fill out non-valid nb cases with zero portion using poisson
    working_nb.loc[~working_nb["valid_nb"],"zero_prop"]=np.exp(-working_nb["mean_umis_mpra_bc"])

    assert ~any(working_nb["zero_prop"].isna())

    #now we need to get the representation of each cre in each replicate

    representation=pd.DataFrame({"rep_id":undone_zi["rep_id"],"cre_id":undone_nb["cre_id"]}).value_counts().reset_index()
    representation=representation.merge(working_nb["zero_prop"].reset_index(),on="cre_id",how="left")

    #computing the number of zeroes we expect for each cre, replicate combination.
    #could alterantively do this as a weighted average, but I feel like this is more readable.

    representation["expected_nb_zeroes"]=representation["zero_prop"]*representation["count"]

    #now condense to a per-replicate summary
    nb_zero=representation.groupby("rep_id").agg(
        total_events=("count","sum"),
        total_zeroes=("expected_nb_zeroes","sum")
    ).reset_index()

    nb_zero["nb_zero_fraction"]=nb_zero["total_zeroes"]/nb_zero["total_events"]
    nb_zero=nb_zero[["rep_id","nb_zero_fraction"]]

    ## subtract zeroes expected from nb portion from actual zeroes to approx the degree of zero inflation##
    zeros=zeros.reset_index().merge(nb_zero,validate="one_to_one")
    zeros["zero_inflation"]=zeros["gross_zeros"]-zeros["nb_zero_fraction"]
    zeros["zero_inflation"]=np.clip(zeros["zero_inflation"],0,1)

    ## compute betas for zi ##
    
    #logistic function
    zi_beta=1/(1 + np.exp(-zeros["zero_inflation"].to_numpy()))
    
    # estimate theta #
    thetas=working_nb["mean_umis_mpra_bc"]**2/(working_nb["var_umis_mpra_bc"]-working_nb["mean_umis_mpra_bc"])
    thetas=thetas[working_nb["valid_nb"]]
    beta_theta=np.log(np.mean(thetas))
    
    # create & return the initalization values dict #
    
    init={}
    init["x_mu"]=nb_betas.to_numpy()
    init["x_pi"]=zi_beta
    init["theta"]=beta_theta

    return init
mom_result=mom(mats)

2025-12-04 17:24:46,131 - distributed.nanny - WARNING - Restarting worker
2025-12-04 17:24:46,139 - distributed.nanny - WARNING - Restarting worker
2025-12-04 17:24:46,229 - distributed.nanny - WARNING - Restarting worker
2025-12-04 17:24:46,286 - distributed.nanny - WARNING - Restarting worker


In [19]:
mom_result['x_mu']

array([ 8.32132871,  7.62087508,  6.38558224,  7.83289014,  8.75128315,
        8.96056099,  8.77144653,  7.38464493,  9.04435621,  6.78957834,
        8.5192327 ,  6.48276099,  8.58678165,  8.58728302,  6.35183846,
        7.22329883,  8.07982089,  8.52787863,  9.34954089,  6.65366317,
        7.68242882,  9.06691145,  8.75055816,  8.74186044,  8.91563564,
        8.75818241,  8.92174444,  8.67980874,  8.97265814,  8.77458022,
        6.33857484,  4.95297373,  7.9143745 ,  9.08134501,  8.62725437,
        8.95531093,  8.40864267,  9.07073346,  8.56635684,  6.54100379,
        9.04365582,  8.80685994,  9.07391656,  8.0823701 ,  7.93004351,
        6.29479627,  9.0257079 ,  9.22662428,  7.0816041 ,  8.56954361,
       -0.2830854 ,  0.01805465, -0.7949085 , -0.72479802, -0.89010595,
       -0.32287193, -0.59920783, -0.64582215, -0.51652506, -0.71775897,
       -0.50661698, -0.94335651, -0.78141412, -1.26359119, -0.5305113 ,
       -0.76940321, -0.34940113, -0.25570889, -0.45833877, -0.29

manually calc reference beta...

In [28]:
dat=scm.scMPRA_data.from_parquet(f"{DATA_ROOT}/{simpath}/scMPRA/0.scmpra").data

In [ ]:
refmean=dat[(dat["cell_type"]=="Cardiomyocytes") & (dat["cre_id"]=="reference")]["umis_mpra_bc"].mean()

In [34]:
refmean

0.03777335984095427

$$e^{\beta_{ref}}=E[ref]\tag{2}$$

In [36]:
np.log(refmean)

-3.2761511909332985